In [1]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# **Transformer**

In [2]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

# **Data Pre-processing**

In [18]:
test_df = pd.read_csv('../data/initial_datasets/speeches/speeches_test.csv')
train_df = pd.read_csv('../data/initial_datasets/speeches/speeches_train.csv')
synthetic_df = pd.read_csv('../data/generated/speeches/control_synthetic_data.csv')

In [19]:
train_df['label'] = train_df['label'].replace({'Republican': -1, 'Democrat': 1})
test_df['label'] = test_df['label'].replace({'Republican': -1, 'Democrat': 1})

/var/folders/lv/pnwq6bmj4tq68bsvy__37qyh0000gn/T/ipykernel_36729/3264621345.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df['label'] = train_df['label'].replace({'Republican': -1, 'Democrat': 1})
/var/folders/lv/pnwq6bmj4tq68bsvy__37qyh0000gn/T/ipykernel_36729/3264621345.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_df['label'] = test_df['label'].replace({'Republican': -1, 'Democrat': 1})


In [20]:
test_df.rename(columns={'text': 'sentences', 'label': 'labels'}, inplace=True)
train_df.rename(columns={'text': 'sentences', 'label': 'labels'}, inplace=True)
synthetic_df.rename(columns={'text': 'sentences', 'label': 'labels'}, inplace=True)

In [21]:
synthetic_df = synthetic_df.sample(n=250, random_state=42)
test_df = test_df.sample(n=250, random_state=42)
train_df = train_df.sample(n=250, random_state=42)

# **Distributional Measures**

In [8]:
import numpy as np
import ot

def wasserstein(X, Y):
    
    a = np.ones((X.shape[0],)) / X.shape[0]
    b = np.ones((Y.shape[0],)) / Y.shape[0]

    M = ot.dist(X, Y)
    M /= M.max()

    return ot.emd2(a, b, M)

In [52]:
original_df = pd.concat([train_df, test_df]).sample(n=500)
synthetic_df = synthetic_df.sample(n=500)

X = np.array(sentence_transformer.encode(original_df['sentences'].to_list()))
Y = np.array(sentence_transformer.encode(synthetic_df['sentences'].to_list()))

In [53]:
def report_MMD(X, Y, normalize=False):
    
    tensorX = torch.tensor(X)
    tensorY = torch.tensor(Y)
    rbf_mmd = MMD(tensorX, tensorY, "rbf")
    scale_mmd = MMD(tensorX, tensorY, "multiscale")


    if normalize:
        return (rbf_mmd.item() / np.sqrt((1.0 / X.shape[0]) + (1.0 / Y.shape[0]))), (scale_mmd.item() / np.sqrt(1.0 / X.shape[0] + 1.0 / Y.shape[0]))
    else:
        return rbf_mmd.item(), scale_mmd.item()

In [54]:
report_MMD(X, Y, True)

(np.float64(1.009706762034249), np.float64(6.316165412070131))

In [55]:
wasserstein(X, Y)

0.4851508805751801

# **Classifier**

In [22]:
X_train = np.array(sentence_transformer.encode(synthetic_df['sentences'].to_list()))
X_test = np.array(sentence_transformer.encode(test_df['sentences'].to_list()))

y_train = synthetic_df['labels']
y_test = test_df['labels']

In [23]:
model = svm.SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_train)
train_acc = accuracy_score(y_pred, y_train)
print(f'Train acc: {train_acc}')

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_pred, y_test)
print(f'Test acc: {test_acc}')

Train acc: 1.0
Test acc: 0.664


In [24]:
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))

0.5352112676056338
0.8085106382978723


In [25]:
y_prob = model.predict_proba(X_test)[:, 1]
print(roc_auc_score(y_test, y_prob))

0.7810624659028914


# **Classifier with 10% train data**

In [26]:
train = pd.concat([train_df.sample(n=25, random_state=42), synthetic_df.sample(n=225, random_state=42)])
X_train = np.array(sentence_transformer.encode(train['sentences'].to_list()))
X_test = np.array(sentence_transformer.encode(test_df['sentences'].to_list()))

y_train = train['labels']
y_test = test_df['labels']

In [27]:
model = svm.SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_train)
train_acc = accuracy_score(y_pred, y_train)
print(f'Train acc: {train_acc}')

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_pred, y_test)
print(f'Test acc: {test_acc}')

Train acc: 1.0
Test acc: 0.728


In [28]:
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))

0.8611111111111112
0.32978723404255317


In [29]:
y_prob = model.predict_proba(X_test)[:, 1]
print(roc_auc_score(y_test, y_prob))

0.8354473540643754
